# Data Cleaning and Feature Engineering

Apply the modular transformation workflow and inspect its audit trail.

**Student:** S. M. Monowar Kayser (253-25-019)  
**Course:** Data Visualization (CSE628), Summer 2026

All paths are project-relative and all reported values are calculated from the stored data or `outputs/analysis_summary.json`.

In [1]:
from pathlib import Path
import json
import sys
from IPython.display import Image, display
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
SUMMARY = json.loads((ROOT / 'outputs' / 'analysis_summary.json').read_text(encoding='utf-8'))
print('Project root resolved successfully.')
print('Canonical summary: outputs/analysis_summary.json')

Project root resolved successfully.
Canonical summary: outputs/analysis_summary.json


In [2]:
from src.config import RAW_CSV
from src.data_loader import load_facebook_data
from src.data_preprocessing import clean_and_engineer, validate_processed
processed, audit = clean_and_engineer(load_facebook_data(RAW_CSV))
validate_processed(processed)
display(audit)
display(processed.head())

{'initial_rows': 7050,
 'initial_columns': 16,
 'empty_columns_removed': ['Column1', 'Column2', 'Column3', 'Column4'],
 'duplicate_rows_removed': 0,
 'duplicate_ids_removed': 0,
 'numeric_missing_values_imputed': 0,
 'invalid_dates_removed': 0,
 'final_rows': 7050,
 'final_columns': 27,
 'outlier_method': 'Upper Tukey fence on total_engagement',
 'outlier_threshold': 1019.0,
 'outlier_count': 972,
 'reaction_gap_nonzero_rows': 9}

,status_id,status_type,status_published,num_reactions,num_comments,num_shares,num_likes,num_loves,num_wows,num_hahas,...,positive_reaction_percentage,negative_reaction_percentage,posting_hour,day_of_week,month,week_number,is_weekend,time_of_day,engagement_category,engagement_outlier
0,2635,photo,2012-07-15 02:51:00,15,3,0,15,0,0,0,...,100.0,0.0,2,Sunday,2012-07,28,True,Overnight,Low,False
1,2634,photo,2012-07-15 02:58:00,14,7,0,14,0,0,0,...,100.0,0.0,2,Sunday,2012-07,28,True,Overnight,Low,False
2,2633,photo,2012-07-15 03:32:00,14,1,0,14,0,0,0,...,100.0,0.0,3,Sunday,2012-07,28,True,Overnight,Low,False
3,2632,photo,2012-07-15 03:42:00,12,3,0,12,0,0,0,...,100.0,0.0,3,Sunday,2012-07,28,True,Overnight,Low,False
4,2631,photo,2012-07-15 03:54:00,19,17,3,19,0,0,0,...,100.0,0.0,3,Sunday,2012-07,28,True,Overnight,Medium,False


In [3]:
engineered = ['reaction_components_total', 'total_engagement', 'like_to_comment_ratio', 'share_to_engagement_ratio', 'positive_reaction_percentage', 'posting_hour', 'day_of_week', 'month', 'is_weekend', 'time_of_day', 'engagement_category', 'engagement_outlier']
display(processed[engineered].describe(include='all').T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
reaction_components_total,7050.0,NaN,NaN,NaN,230.114468,462.624061,0.0,17.0,59.5,219.0,4710.0
total_engagement,7050.0,NaN,NaN,NaN,494.495745,1152.169364,0.0,19.0,69.0,419.0,21708.0
like_to_comment_ratio,7050.0,NaN,NaN,NaN,19.852743,73.623824,0.0,0.0,1.930952,14.857143,2300.0
share_to_engagement_ratio,7050.0,NaN,NaN,NaN,0.039735,0.085852,0.0,0.0,0.0,0.037037,1.0
positive_reaction_percentage,7050.0,NaN,NaN,NaN,97.821294,13.042612,0.0,100.0,100.0,100.0,100.0
posting_hour,7050.0,NaN,NaN,NaN,7.829504,6.886893,0.0,2.0,7.0,9.0,23.0
day_of_week,7050,7,Sunday,1059,NaN,NaN,NaN,NaN,NaN,NaN,NaN
month,7050,72,2017-12,669,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_weekend,7050,2,False,5009,NaN,NaN,NaN,NaN,NaN,NaN,NaN
time_of_day,7050,5,Overnight,2998,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Outliers are flagged rather than deleted. Ratio calculations are zero-safe, and no engagement-rate claim is made because reach and impression denominators are absent.